## 1) Importar bibliotecas

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import polars as pl
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## 2) Ler base de dados

In [ ]:
train_df = pl.read_csv(source = "./dados/train_quadratic.csv")

train_df = train_df.with_columns(
    pl.lit(
        np.random.random(
            size = train_df.shape[0]
        )
    ).alias("order")
)

train_df = train_df.sort(
    by = ["order"]
).drop("order")

print(train_df.shape)
train_df.head(2)

### 2.1) Ver quantidade de classes

In [ ]:
display(
    train_df.group_by("class").agg(pl.col("class").len().alias("total"))
)

### 2.2) Ver distribuição por classe

In [ ]:
fig, ax = plt.subplots()

ax.scatter(
    x = train_df["x"],
    y = train_df["y"],
    c = train_df["class"],
    edgecolors = "black"
)
ax.set_title("Distribuição dos pontos segundo classes")

plt.show()

### 2.3) Separar dataframe em numpy

In [ ]:
X = train_df[["x", "y"]].to_numpy()
y = train_df["class"].to_numpy()

print(X[:5])
print(y[:5])

## 3) Criar modelo

In [ ]:
X.shape

### 3.1) Definindo parâmetros

In [ ]:
neurons = 5

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape = (2,)),
    tf.keras.layers.Dense(units = 2**neurons, activation = "relu"),
    tf.keras.layers.Dense(units = 2**neurons, activation = "relu"),
    tf.keras.layers.Dense(units = 2, activation = "softmax"),
])

model_optimizer = tf.keras.optimizers.Adam(learning_rate = 1E-4)
model_loss = tf.keras.losses.SparseCategoricalCrossentropy()
model_metrics = tf.keras.metrics.SparseCategoricalAccuracy()

model.compile(
    optimizer = model_optimizer,
    loss = model_loss,
    metrics = [model_metrics],
)

model.summary()

### 3.2) Configurando <code>early_stopping</code>

In [ ]:
model_callback_stopping = tf.keras.callbacks.EarlyStopping(
    min_delta = 1E-2,
    patience = 5,
    verbose = 1,
    start_from_epoch = 100
)

### 3.3) Treinando modelo

In [ ]:
history = model.fit(
    x = X,
    y = y,
    batch_size = 16,
    validation_split = 0.2,
    shuffle = True,
    epochs = 100,
    callbacks = [model_callback_stopping]
)

### 3.4) Avaliando treinamento

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    history.epoch,
    history.history["val_loss"],
    "C0",
    label = "val_loss"
)

ax.plot(
    history.epoch,
    history.history["loss"],
    "C1",
    label = "loss"
)

ax.set_title("Evolução da loss em meio treinamento.")

plt.legend()
plt.show()

### 3.5) Avaliando predição

In [ ]:
X_test = pl.read_csv("./dados/test_quadratic.csv")

print(X_test.shape)
X_test.head(2)

In [ ]:
fig, axs = plt.subplots(
    ncols = 2,
    figsize = (16, 5)
)

ax1, ax2 = axs[0], axs[1]
predictions = model.predict(X_test[["x", "y"]].to_numpy())
prediction = predictions.argmax(axis = 1)

ax1.scatter(
    x = X_test["x"],
    y = X_test["y"],
    c = X_test["class"],
    edgecolors = "black",
)

ax2.scatter(
    x = X_test["x"],
    y = X_test["y"],
    c = prediction,
    edgecolors = "black"
)

ax1.set_title("Real classification")
ax2.set_title("Prediction by model")

plt.suptitle("Distribution", fontsize = 16)
plt.show()

fig, axs = plt.subplots(
    ncols = 2,
    figsize = (16, 5)
)

axs[0].hist(
    predictions
)

confusion_matrix_plot = ConfusionMatrixDisplay(
    confusion_matrix = confusion_matrix(y_true= X_test["class"], y_pred=prediction),
)

confusion_matrix_plot.plot(
    ax = axs[1],
    
)
axs[0].set_title("Model class priority distribution")
axs[1].set_title("Model performance")

plt.show()